<a href="https://colab.research.google.com/github/snatzehua/bt-4012-project-2026/blob/feat--zs-exploration-00/notebooks/data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BT4012 · Shared data preparation

**Purpose:** create the common, unencoded input for baseline models and future experiments. This notebook is standalone: it does not import a repository `src/` package.

It reads your existing `Base.csv`, decodes documented negative missing-value sentinels, assigns stable source-row IDs, and saves `baf_clean.parquet` plus `preparation_manifest.json`. The raw CSV is never changed. Re-running replaces these two prepared artifacts; keep a prior copy if you need to preserve an earlier preparation version.

**What is deliberately left to modeling:** imputation, scaling, categorical vocabulary fitting, feature transformations and resampling. Those must be fitted separately on each actual training subset, including Recent K and Historical K.

Run all cells in order. A CPU runtime is sufficient. Colab and local paths are configured in Section 1. This notebook does not replace or modify `eda_zs.ipynb`.


## 1. Configure the runtime and file locations

Your current locations are supported directly: Drive `BT4012_grp_proj/Base.csv`, or local `<repo>/data/Base.csv`.


In [ ]:
import sys
from pathlib import Path

# Detect the runtime; local execution does not require google.colab.
IN_COLAB = 'google.colab' in sys.modules

# Colab: keep Base.csv directly inside this existing Drive project folder.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/BT4012_grp_proj')

# Local: normally leave None; set an absolute repo path if auto-detection fails.
LOCAL_REPO_ROOT = None

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_DIR = DRIVE_PROJECT_DIR
    RAW_PATH = PROJECT_DIR / 'Base.csv'
    PROCESSED_DIR = PROJECT_DIR / 'processed'
else:
    if LOCAL_REPO_ROOT is not None:
        PROJECT_DIR = Path(LOCAL_REPO_ROOT).expanduser().resolve()
    else:
        working_dir = Path.cwd().resolve()
        candidates = [working_dir, *working_dir.parents]
        PROJECT_DIR = next(
            (
                candidate
                for candidate in candidates
                if (candidate / 'notebooks').is_dir()
                and (candidate / 'data').is_dir()
            ),
            working_dir,
        )

    RAW_PATH = PROJECT_DIR / 'data' / 'Base.csv'
    PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

CLEAN_PATH = PROCESSED_DIR / 'baf_clean.parquet'
MANIFEST_PATH = PROCESSED_DIR / 'preparation_manifest.json'

print('Runtime:', 'Google Colab' if IN_COLAB else 'Local Python')
print('Project:', PROJECT_DIR)
print('Raw CSV:', RAW_PATH)
print('Prepared data:', CLEAN_PATH)


Mounted at /content/drive
Runtime: Google Colab
Project: /content/drive/MyDrive/BT4012_grp_proj
Raw CSV: /content/drive/MyDrive/BT4012_grp_proj/Base.csv
Prepared data: /content/drive/MyDrive/BT4012_grp_proj/processed/baf_clean.parquet


## 2. Check dependencies

Colab installs only a missing Parquet engine. Locally, install `requirements.txt` in your chosen environment before running; this cell does not alter a local environment automatically.


In [ ]:
import importlib.util
import subprocess

# Parquet is the shared data format; pandas needs an installed engine to write it.
if importlib.util.find_spec('pyarrow') is None:
    if IN_COLAB:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow>=14.0',
        ])
    else:
        raise ImportError('Missing pyarrow. Run: python -m pip install -r requirements.txt')

import hashlib
import json
import platform
from datetime import datetime, timezone
from uuid import uuid4

import numpy as np
import pandas as pd
import pyarrow
from IPython.display import display


## 3. Declare the dataset contract

Use months 0–4 for training, month 5 for validation, and months 6–7 for testing. Category names stay as raw strings; numeric codes and one-hot columns are created later within model pipelines.


In [ ]:
TARGET = 'fraud_bool'
MONTH = 'month'
ROW_ID = 'row_id'

TRAIN_MONTHS = [0, 1, 2, 3, 4]
VAL_MONTHS = [5]
TEST_MONTHS = [6, 7]

CATEGORICAL = [
    'payment_type',
    'employment_status',
    'housing_status',
    'source',
    'device_os',
]

MISSING_IF_NEGATIVE = [
    'prev_address_months_count',
    'current_address_months_count',
    'intended_balcon_amount',
    'bank_months_count',
    'session_length_in_minutes',
    'device_distinct_emails_8w',
]

EXPECTED_COLUMNS = [
    TARGET,
    'income',
    'name_email_similarity',
    'prev_address_months_count',
    'current_address_months_count',
    'customer_age',
    'days_since_request',
    'intended_balcon_amount',
    'payment_type',
    'zip_count_4w',
    'velocity_6h',
    'velocity_24h',
    'velocity_4w',
    'bank_branch_count_8w',
    'date_of_birth_distinct_emails_4w',
    'employment_status',
    'credit_risk_score',
    'email_is_free',
    'housing_status',
    'phone_home_valid',
    'phone_mobile_valid',
    'bank_months_count',
    'has_other_cards',
    'proposed_credit_limit',
    'foreign_request',
    'source',
    'session_length_in_minutes',
    'device_os',
    'keep_alive_session',
    'device_distinct_emails_8w',
    'device_fraud_count',
    MONTH,
]


def file_sha256(path):
    """
    Calculate a file fingerprint without loading the complete file into memory.

    Read the bytes in chunks and return a SHA-256 hexadecimal digest. The manifest
    uses this to identify the exact source and prepared files used by later runs.
    """
    digest = hashlib.sha256()

    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)

    return digest.hexdigest()


## 4. Load and validate Base.csv

Fail explicitly on a different schema rather than silently admitting an exported index or unrelated columns. Exact duplicate rows are reported but are not automatically deleted. No holdout fraud rates are displayed.


In [ ]:
if not RAW_PATH.is_file():
    raise FileNotFoundError(f'Cannot find {RAW_PATH}. Check the paths in Section 1.')

raw = pd.read_csv(RAW_PATH)

# Match columns by name; retain the source order for reproducible row IDs.
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(raw.columns))
extra_columns = sorted(set(raw.columns) - set(EXPECTED_COLUMNS))

if missing_columns or extra_columns:
    raise ValueError(
        f'Unexpected CSV schema. Missing: {missing_columns}; extra: {extra_columns}. '
        'Investigate the file before changing the expected schema.'
    )

if raw[[TARGET, MONTH]].isna().any().any():
    raise ValueError('Target/month contains missing values.')

if set(raw[TARGET].unique()) != {0, 1}:
    raise ValueError('Expected both binary target labels 0 and 1.')

if set(raw[MONTH].unique()) != set(range(8)):
    raise ValueError('Expected month indices 0–7 for the complete Base dataset.')

numeric_columns = [column for column in EXPECTED_COLUMNS if column not in CATEGORICAL]

if not all(pd.api.types.is_numeric_dtype(raw[column]) for column in numeric_columns):
    raise TypeError('An expected numeric column was read as text; inspect the CSV.')

if np.isinf(raw[numeric_columns].to_numpy(dtype=float)).any():
    raise ValueError('Infinite numeric values found; investigate before modeling.')

quality = pd.Series({
    'applications': len(raw),
    'source_columns': raw.shape[1],
    'exact_duplicate_rows': int(raw.duplicated().sum()),
    'literal_missing_cells': int(raw.isna().sum().sum()),
}, name='value')

display(quality.to_frame())


,value
applications,1000000
source_columns,32
exact_duplicate_rows,0
literal_missing_cells,0


## 5. Apply common transformations

Convert negative values to `NaN` only in the six documented sentinel columns. In particular, **all** negative `intended_balcon_amount` values encode missingness. Do not globally replace negative values in other features.

`row_id` is the zero-based position in this exact source CSV, assigned before any filtering or sampling. It is an alignment key, not a predictor or evidence of chronological order within a month. IDs must be interpreted alongside the source fingerprint if the source file changes.


In [ ]:
# Keep raw unchanged and add an identifier before splitting or sampling.
clean = raw.copy()
clean.insert(0, ROW_ID, np.arange(len(clean), dtype=np.int64))

cleaning_log = []

for column in MISSING_IF_NEGATIVE:
    sentinel_mask = clean[column].lt(0)

    cleaning_log.append({
        'feature': column,
        'negative_sentinels_replaced': int(sentinel_mask.sum()),
        'literal_missing_before': int(clean[column].isna().sum()),
    })

    clean[column] = clean[column].mask(sentinel_mask)

# Preserve labels as strings rather than fitting a vocabulary on all months.
# Object dtype plus np.nan is compatible with the later sklearn encoders.
for column in CATEGORICAL:
    clean[column] = clean[column].map(
        lambda value: str(value) if pd.notna(value) else np.nan
    )

clean[TARGET] = clean[TARGET].astype('int8')
clean[MONTH] = clean[MONTH].astype('int8')

display(pd.DataFrame(cleaning_log))


,feature,negative_sentinels_replaced,literal_missing_before
0,prev_address_months_count,712920,0
1,current_address_months_count,4254,0
2,intended_balcon_amount,742523,0
3,bank_months_count,253635,0
4,session_length_in_minutes,2015,0
5,device_distinct_emails_8w,359,0


## 6. Establish split metadata and the predictor list

Identify constants using months 0–4 only. Retain every column in the prepared file for auditing, but exclude training constants, `row_id`, `month` and the target from the predictor list.

This initial predictor list can stay fixed across later training-window comparisons. A feature becoming constant or entirely missing within a smaller training subset is handled inside its fitted pipeline; its imputation/scaling/category vocabulary must never be borrowed from All history.


In [ ]:
training_rows = clean[MONTH].isin(TRAIN_MONTHS)
feature_candidates = [
    column
    for column in EXPECTED_COLUMNS
    if column not in [TARGET, MONTH]
]

constant_features = [
    column
    for column in feature_candidates
    if clean.loc[training_rows, column].nunique(dropna=False) <= 1
]

features = [
    column for column in feature_candidates if column not in constant_features
]
categorical_features = [column for column in CATEGORICAL if column in features]
numerical_features = [column for column in features if column not in categorical_features]

split_months = {
    'train': TRAIN_MONTHS,
    'validation': VAL_MONTHS,
    'test': TEST_MONTHS,
}

split_counts = {
    split: int(clean[MONTH].isin(months).sum())
    for split, months in split_months.items()
}

print('Excluded training constants:', constant_features)
print(f'Candidate predictors: {len(features)}')
display(pd.DataFrame([
    {'split': split, 'months': months, 'applications': split_counts[split]}
    for split, months in split_months.items()
]))


Excluded training constants: ['device_fraud_count']
Candidate predictors: 29


,split,months,applications
0,train,"[0, 1, 2, 3, 4]",675666
1,validation,[5],119323
2,test,"[6, 7]",205011


## 7. Save the reusable artifacts

The Parquet file stores all cleaned applications, including the split key and labels. The manifest records the contract; it is **not** a fitted preprocessor. Keep both files together. Outputs are first written to temporary files before replacing the existing pair; a checksum detects mismatches if a run is interrupted.


In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Write the data first, then record its exact fingerprint in the manifest.
parquet_temp = PROCESSED_DIR / 'baf_clean.tmp.parquet'
clean.to_parquet(parquet_temp, index=False, engine='pyarrow', compression='snappy')
processed_sha256 = file_sha256(parquet_temp)

manifest = {
    'schema_version': 1,
    'preparation_id': uuid4().hex,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'source_filename': RAW_PATH.name,
    'source_sha256': file_sha256(RAW_PATH),
    'prepared_filename': CLEAN_PATH.name,
    'prepared_sha256': processed_sha256,
    'rows': len(clean),
    'columns': list(clean.columns),
    'dtypes': {column: str(dtype) for column, dtype in clean.dtypes.items()},
    'target': TARGET,
    'month_column': MONTH,
    'row_id_column': ROW_ID,
    'row_id_definition': 'Zero-based source CSV row position before filtering/sampling',
    'split_months': split_months,
    'split_counts': split_counts,
    'features': features,
    'numerical_features': numerical_features,
    'categorical_features': categorical_features,
    'excluded_constant_features': constant_features,
    'negative_missing_columns': MISSING_IF_NEGATIVE,
    'cleaning_log': cleaning_log,
    'training_fraud_rate': float(clean.loc[training_rows, TARGET].mean()),
    'versions': {
        'python': platform.python_version(),
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'pyarrow': pyarrow.__version__,
    },
}

manifest_temp = PROCESSED_DIR / 'preparation_manifest.tmp.json'

with open(manifest_temp, 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

parquet_temp.replace(CLEAN_PATH)
manifest_temp.replace(MANIFEST_PATH)

print('Saved:', CLEAN_PATH)
print('Saved:', MANIFEST_PATH)
print('Preparation ID:', manifest['preparation_id'])
print('Next: run 02_baseline_model_comparison.ipynb.')


Saved: /content/drive/MyDrive/BT4012_grp_proj/processed/baf_clean.parquet
Saved: /content/drive/MyDrive/BT4012_grp_proj/processed/preparation_manifest.json
Preparation ID: 7d68edd15bcf4aea972f0c94dc370c3d
Next: run 02_baseline_model_comparison.ipynb.


## Sources and handoff notes

- [BAF repository](https://github.com/feedzai/bank-account-fraud)
- [Official datasheet](https://github.com/feedzai/bank-account-fraud/blob/main/documents/datasheet.pdf)

The prepared artifacts can be read by notebooks or Python scripts with `pd.read_parquet()` and `json.load()`. They contain no fitted imputation, scaling or encoding. Acknowledge AI assistance in accordance with the project guidelines; these cells do not constitute real-data findings until executed.
